<a href="https://colab.research.google.com/github/ilhamilha-creator/flyrank-ml-assignments/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ilhamilha-creator/flyrank-ml-assignments/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Refresh / Content Opportunity Scoring is a **ranking / scoring** task. The decision is which pages to review first. The model uses a binary observed label (`trend_direction == "down"`), then ranks pages. Success is Precision@50, not whole-catalog accuracy.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
task_type = "Ranking / scoring"
label = "trend_direction == down (observed proxy)"
metric = "Precision@50"

print(f"Project Framework Selected: {task_type}")
print(f"Label: {label}")
print(f"Success metric: {metric}")

Project Framework Selected: Binary Classification
Target Output Definition: {0: 'Stable/Growing', 1: 'Declining/Needs Refresh'}


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The model predicts a proxy label derived from observed performance trends. The target is `trend_direction == "down"`, which indicates whether a page's search impressions have declined in the recent 90-day period compared to historical performance. This is a measured proxy for "needs refresh" - pages showing decline patterns are candidates for content review and updates.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import pandas as pd

csv_path = None
for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    candidate = p / "data" / "raw" / "content_refresh_anonymized.csv"
    if candidate.exists():
        csv_path = candidate
        break
if csv_path is None:
    raise FileNotFoundError("content_refresh_anonymized.csv not found")

df = pd.read_csv(csv_path)

# Create the actual target label
df['is_declining'] = (df["trend_direction"].str.lower() == "down").astype(int)

print("Sample representation of target classification vector (y):")
print(df['is_declining'].head(10).values)
print(f"\nTarget distribution: {df['is_declining'].value_counts().to_dict()}")

Sample representation of target classification vector (y):
[1 1 1 0 1 1 1 0 1 1]

Target distribution: {1: 16262, 0: 13738}


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The metric is Precision@50. Teams can only review a short list, so the question is: of the 50 pages we put at the top, how many actually declined next month (Apr impressions < 80% of Mar)? Random picking on fold 1 is 0.439. A fair hand-written rule (no label in the score) is 0.640. The forest on that fold is 0.280 — it does not beat the rule.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import json
import os

metrics_paths = [
    "work/outputs/canonical_metrics.json",
    "../outputs/canonical_metrics.json",
    "outputs/canonical_metrics.json",
]
metrics = None
for path in metrics_paths:
    if os.path.exists(path):
        with open(path) as f:
            metrics = json.load(f)
        break

if metrics is None:
    raise FileNotFoundError("canonical_metrics.json not found")

print("Success metric: Precision@50")
print(f"Base rate (random): {metrics['base_rate']:.3f}")
print(f"Fair baseline: {metrics['fair_baseline_precision_at_50']:.3f}")
print(f"Random Forest: {metrics['random_forest_precision_at_50']:.3f}")
print(f"Model vs fair baseline: {metrics['model_vs_baseline_ratio']:.2f}x")
print(f"Model vs random: {metrics['model_vs_random_ratio']:.2f}x")

Success metric: Precision@50
Base rate (random): 0.439
Fair baseline: 0.640
Random Forest: 0.280
Model vs fair baseline: 0.44x
Model vs random: 0.64x


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is one unique content page. Each row represents a single content item with its aggregated search performance features over the 90-day analysis window. This grain matches the decision: content teams review individual pages, not domains or queries.

In [4]:
from pathlib import Path
import pandas as pd

csv_path = None
for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    candidate = p / "data" / "raw" / "content_refresh_anonymized.csv"
    if candidate.exists():
        csv_path = candidate
        break
if csv_path is None:
    raise FileNotFoundError("content_refresh_anonymized.csv not found")

df = pd.read_csv(csv_path)

print("Dataset loaded successfully.")
print(f"Unit of Analysis: {df.shape[0]:,} unique content pages (rows)")
print(f"Features per page: {df.shape[1]} performance signals (columns)")

# Show the first few columns to represent the unit of analysis
print("\nSample of unit of analysis (first 2 pages):")
print(df.iloc[:, :5].head(2))

Dataset loaded successfully.
Unit of Analysis: 30,000 unique content pages (rows)
Features per page: 44 performance signals (columns)

Sample of unit of analysis (first 2 pages):
             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  content_a1fb4e703a9e  client_4e07408562           90.0         0.01   

  competition_level  
0              HIGH  
1               LOW  


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A single if-statement is too blunt. "Refresh if older than 180 days" treats a stable old page the same as a declining one. Age, freshness, impressions, CTR, and average position show up together, and the mix is messy enough that a weighted model can rank better than one cutoff.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import pandas as pd

csv_path = None
for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    candidate = p / "data" / "raw" / "content_refresh_anonymized.csv"
    if candidate.exists():
        csv_path = candidate
        break
if csv_path is None:
    raise FileNotFoundError("content_refresh_anonymized.csv not found")

df = pd.read_csv(csv_path)

print("Why a single cutoff is weak:")
print("\n1. Rule: 'refresh if content_age_days > 180'")
old_stable = df[(df['content_age_days'] > 180) & (df['trend_direction'] == 'up')]
old_declining = df[(df['content_age_days'] > 180) & (df['trend_direction'] == 'down')]
print(f"   Old but up: {len(old_stable)}")
print(f"   Old and down: {len(old_declining)}")

print("\n2. Age alone mixes those two groups. The model can weight age with impressions, CTR, and average position.")

Why a single cutoff is weak:

1. Rule: 'refresh if content_age_days > 180'
   Old but up: 2928
   Old and down: 8564

2. Age alone mixes those two groups. The model can weight age with impressions, CTR, and average position.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.